# The optical depth of skewers through a simulation box

The counterpart of `6_optical_depth.ipynb` for a real simulation. Instead of hand-defined toy fields, skewers are thrown through a hydrodynamical box and SkewerJAX computes their optical depth τ and transmitted flux F = e^(−τ).

**How the fields are made.** The companion notebook `3_fields_from_simulations.ipynb` builds the fields along the skewers: deposited pieces of Voronoi cells, each with its own density, temperature, peculiar velocity and length. This notebook takes them as given and focuses on the integration:
- the role of the sampling points and of the line elements;
- the cosmology inside the integral, through the scale factor along the path;
- the spectral grid and the periodicity of a box;
- the resolution criterion and its warning.

Everything is SkewerJAX's default, which is also its most accurate mode.

**Data.** TNG50-4, snapshot 025 (z = 3.008), a periodic box of 51.7 cMpc. H I (Lyα) first, then a short section on Si III 1206.5 Å.

**Runtime.** About 3 min on a CPU from an empty cache; under a minute with `CACHE`.

## 0. Setup

In [ ]:
import os, sys, time, warnings
os.environ.setdefault("JAX_PLATFORMS", "cpu")             # float64 runs faster on this CPU than on a consumer GPU
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
from pathlib import Path
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib as mpl
import matplotlib.pyplot as plt
from skewerjax.config.setup_matplotlib import (matplotlib_tol_config, TOL_BRIGHT as TOL, ENTITY_COLORS as EC, MODE_STYLES as MS,
                                               tol_cmap, add_panel_letters, residual_band)
font, rcnew = matplotlib_tol_config()                         # the package style with Paul Tol's colour-blind-safe palette
plt.rc("font", **font); plt.rcParams.update(rcnew)

# ------------------------------------------------------------------ configuration (edit here)
REPO = Path("/home/dlopez/Documentos/0.profesional/Postdoc/USP/Projects/SkewerJAX")
DATA = REPO / "claude_agents/tmp_files/20261021_1335"        # caches and fake_spectra reference data (to be moved: see the notebook header)
SNAP_BASE, SNAPNUM = "/home/STORAGE/TNG50-4", 25          # read-only
CACHE = DATA / "cache" / "sjx_cache.h5"                    # SkewerJAX tau results (None: no cache)
DS_KMS = 0.25                                                # spacing of the sampling points, in Hubble velocity [km/s]
RES = 1.0                                                    # spectral pixel [km/s]
T_START = time.perf_counter()
from skewerjax.simulations import snapshot_readers as sr, gas_physics as gp, cell_selection as sel
from skewerjax.simulations import simulation_skewers as ss
from skewerjax.fields import deposition as dep
from skewerjax.optical_depth import absorbers as ab_
from skewerjax.optical_depth.optical_depth import OpticalDepthCalculator
from skewerjax.skewers.skewers import SkewerBundle, path_geometry, axis_aligned_skewer_points
from skewerjax.constants.lines import LineBundle
from skewerjax.constants.physical_constants import phys
print("jax", jax.__version__, jax.devices()[0], "| x64", jax.config.jax_enable_x64)

RESULTS = []
def check(name, value, tol, section, kind="pass", detail=""):
    """kind="pass": PASS if value <= tol; "explained": EXPLAINED if value <= tol (a difference accounted
    for by its parts); "expect": PASS if value > tol (the test must detect something)."""
    ok = bool(value > tol) if kind == "expect" else bool(value <= tol)
    tag = {"pass": ("PASS", "FAIL"), "expect": ("PASS", "FAIL"), "explained": ("EXPLAINED", "UNEXPLAINED")}[kind][0 if ok else 1]
    RESULTS.append(dict(section=section, name=name, value=float(value), tol=tol, kind=kind, tag=tag, detail=detail))
    print(f"[{tag}] {section:5s} {name}: {value:.3e} ({'>' if kind == 'expect' else '<='} {tol:g}) {detail}")
def rel(a, b, mask=None):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.ones(a.shape, bool) if mask is None else np.asarray(mask, bool)
    return np.abs(a[m] - b[m]) / np.maximum(np.abs(b[m]), 1e-300)

In [ ]:
rd = sr.ArepoHDF5Reader(SNAP_BASE, SNAPNUM); H = rd.header; L = H.box_cMpc
LYA = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
SETS = dict(single=dict(axis=np.array([3], dtype=np.int32), cofm=np.array([[17500.25, 17500.75, 0.0]])),
            nset=dict(axis=np.tile(np.array([1, 2, 3], dtype=np.int32), 6)[:16], cofm=np.random.default_rng(2155).uniform(0.0, 35000.0, size=(16, 3))))
print(f"TNG50-4 snapshot {SNAPNUM}: z = {H.redshift:.6f}, a_snap = {H.a:.8f}, box L = {L:.4f} cMpc")

## 1. The integral, discretised on the pieces

For a line j of species i (wavelength λ_j, oscillator strength f_j, Einstein coefficient A_j), the optical depth at the spectral coordinate Λ is

$$\tau(\Lambda)=\mathcal C_j\int_\gamma \frac{n_{\rm com}}{a^3}\,\phi\,dl_{\rm phys}=\mathcal C_j\int_\gamma \frac{n_{\rm com}}{a^2}\,\phi\,dl_{\rm com}\;\simeq\;\mathcal C_j\sum_i\sum_{p\in i}\frac{n_{{\rm com},p}\,\ell_p}{a_p^{2}}\;\phi_p(\Lambda),\qquad \mathcal C_j=\frac{\pi e^2}{m_ec}f_j .$$

Here i runs over the sampling points, p over the pieces of cells inside the dual segment of point i, and ℓ_p is the comoving length of piece p, its **line element**.
- **The weights.** n_com,p ℓ_p a_p⁻² is the physical column of the piece at its own epoch. These are the weights of the quadrature; the trapezoid weights of the points are not used.
- **The profile** φ_p is the Voigt profile of the piece's temperature, with its natural damping, at the frequency offset that the piece sees:

$$\nu_p(\Lambda)=\frac{c}{\lambda_j}\,\mathrm{expm1}\!\Big(-\frac{\Lambda-\Lambda_{c,p}}{c}\Big),\qquad \Lambda_{c,p}=c\ln\frac{a_\dagger}{a(s_p)}+\frac{c}{2}\ln\frac{1+\beta_p}{1-\beta_p},\qquad \beta_p=\frac{v_{{\rm pec},p}\cdot\hat t}{c}.$$

Λ_c,p is the line centre of the piece. Its first term is the cosmological redshift between the reference scale factor a_† and the piece's own a(s_p); the second is the relativistic Doppler shift of its peculiar velocity. The expm1 form is exact, and it keeps full precision when Λ ≈ Λ_c, also in float32.

**What each ingredient controls:**

| ingredient | where it comes from | what it sets |
|---|---|---|
| sampling points | the `SkewerBundle` of the skewers | the segments, hence where each piece's absorption is placed (its centroid) |
| line elements ℓ_p | the deposition | the column of every piece: exact for any spacing |
| a(s) | the cosmology, once the box is placed between emitter and observer | the line centres, and the physical density n_com/a³ |
| spectral grid | free | where τ is evaluated; periodic for a box |

## 2. Skewers through the box

A skewer is any polyline through the box, given as the points of a `SkewerBundle`. `SimulationSkewers` takes those points and selects the cells near them (layers 1–5).
- `SimulationSkewers(reader, points)` takes any paths: points of shape (N_sk, N, 3) in comoving Mpc, unwrapped along each path.
- `SimulationSkewers.axis_aligned(reader, cofm, axis, spacing_kms)` builds straight skewers along the box axes, sampled every `spacing_kms` of Hubble flow.

Here: one skewer along z ("single") and 16 skewers along x, y and z in turn at random positions ("nset"), all sampled every 0.25 km/s.

In [ ]:
SK = {name: ss.SimulationSkewers.axis_aligned(rd, cfg["cofm"], cfg["axis"], DS_KMS, name=name, cache=CACHE) for name, cfg in SETS.items()}
for name, S in SK.items():
    print(f"{name:6s}: {S.n_skewers:2d} skewers x {S.points.shape[1]} points, spacing {np.median(np.diff(S.s[0])) * 1e3:.2f} ckpc | "
          f"{len(S.cells):,} cells selected in {S.selection_s:.1f} s")
A1 = SK["single"].absorbers(gp.HI)
print(f"single skewer: {int((A1.n[0] > 0).sum()):,} pieces; their lengths sum to {A1.length[0].sum():.6f} cMpc (L = {L:.6f})")

## 3. The cosmology inside the integral: the frame

**Placing the box.** The box is placed along the line of sight between the emitter and the observer. The observer is before the first point, so photons travel towards decreasing path length s. The scale factor of every point then follows from the cosmology: a(s) solves

$$\frac{da}{d\chi}=\frac{a^2H(a)}{c},\qquad \chi=s_{\rm ref}-s\ \text{(comoving distance towards the observer)},$$

anchored so that the snapshot's own scale factor sits at the centre of the box, a(L/2) = a_snap. This is the `ExactFrame` (`SimulationSkewers.frame("exact")`), with:
- a_† = a(0), so that a static absorber at the near end of the box sits at Λ = 0;
- the box length in velocity P = c ln(a(0)/a(L)): the period of the spectrum.

**What changes along the box.**
- **The line centre** of a static absorber is c ln(a_†/a(s)). This is nearly, but not exactly, linear in s.
- **The density.** The comoving density of every piece is conserved. Its physical density is n_com/a(s)³ and its physical length a(s) ℓ, so its weight in τ is n_com ℓ a(s)⁻²: the gas at the near end (larger a) absorbs slightly less than the same gas at the far end.

The frame encodes all of the cosmology: neither the Hubble flow nor a "box redshift" is an input field.

In [ ]:
S = SK["single"]; fr = S.frame("exact"); s = S.s[0]; a = fr.a_grid[0]
v_stat = phys.c_kms * np.log(fr.a_dagger / a)
lin = H.velfac_kms_per_cMpc * s
print(f"a(0) = {a[0]:.8f}, a(L/2) = {np.interp(L / 2, s, a):.8f} (a_snap = {H.a:.8f}), a(L) = {a[-1]:.8f} | a_dagger = {fr.a_dagger:.8f}")
print(f"period P = c ln(a(0)/a(L)) = {fr.period_kms:.4f} km/s (linear Hubble flow a H L = {H.vmax_kms:.4f} km/s)")
check("a(L/2) = a_snap", abs(np.interp(L / 2, s, a) / H.a - 1), 1e-10, "3")
check("the period is the line-centre span of the box", abs(v_stat[-1] - v_stat[0] - fr.period_kms) / fr.period_kms, 1e-10, "3")
fig, axs = plt.subplots(1, 3, figsize=(21, 6.2), layout="constrained")
axs[0].plot(s, (a / H.a - 1) * 100, color="k", lw=2.5); axs[0].axvline(L / 2, color=TOL["grey"], ls="--", lw=2)
axs[0].annotate(r"$a = a_{\rm snap}$ at the centre", xy=(L / 2, 0), xytext=(L / 2 + 3, 0.6), fontsize=15, arrowprops=dict(arrowstyle="->"))
axs[0].set(xlabel="path length s [cMpc]", ylabel=r"$a(s)/a_{\rm snap}-1$ [%]", title="Scale factor: −1.3% across the box")
axs[1].plot(s, v_stat - lin, color=EC["HI"], lw=2.5); axs[1].axhline(0, color="k", lw=1)
j = np.argmin(v_stat - lin)
axs[1].annotate(f"{(v_stat - lin)[j]:.1f} km/s", xy=(s[j], (v_stat - lin)[j]), xytext=(s[j] - 5, (v_stat - lin)[j] * 0.45), fontsize=15, arrowprops=dict(arrowstyle="->"))
axs[1].set(xlabel="path length s [cMpc]", ylabel=r"$c\ln(a_\dagger/a)-aH\,s$ [km/s]", title="Line centre minus a linear map")
axs[2].plot(s, ((H.a / a) ** 2 - 1) * 100, color=EC["density"], lw=2.5); axs[2].axhline(0, color="k", lw=1)
axs[2].set(xlabel="path length s [cMpc]", ylabel=r"$(a_{\rm snap}/a)^2-1$ [%]", title="Weight of the same gas: ±1.3%")
add_panel_letters(axs)
plt.show()

*Figure — the frame.* Left: a(s) falls by 1.3% across the box. Middle: the line centre of a static absorber departs from a linear map by up to ~3 km/s at the centre, and the departure vanishes at the two ends. Right: the a⁻² weight changes the absorption of the same comoving gas by ±1.3% between the two ends.

## 4. The spectral grid, and the periodicity of a box

**The grid is free.** τ can be evaluated at any set of Λ, independently of the sampling points: they are different discretisations. For a periodic box, the natural grid is one period, [0, P), in pixels of width Δv = P/n with n = int(P/1 km/s): `SimulationSkewers.spectral_grid(1.0, frame)`.

**Periodicity.** A periodic box has a periodic spectrum: the gas just beyond the far face is the gas at the near face. SkewerJAX makes the integral periodic by giving every piece the **nearest periodic image** of its detuning:

$$\Lambda-\Lambda_{c,p}\ \longrightarrow\ \big[(\Lambda-\Lambda_{c,p}+P/2)\bmod P\big]-P/2 .$$

This is the `velocity_period = P` option of `OpticalDepthCalculator`, which the frame sets. It costs nothing (one `mod`) and needs no extra points. It differs from the exact sum over all images only by the far wings beyond P/2 of each piece. Those are negligible except for damped absorbers, whose wings are then truncated half a box away.

**A non-periodic path** (a finite sightline, a light cone) instead needs `periodic=False` and a grid that covers its line centres plus the line widths (§8).

In [ ]:
v, dv = S.spectral_grid(RES, fr)
print(f"spectral grid: {v.size} pixels of {dv:.6f} km/s over one period [0, {v.size * dv:.4f})")
uu = np.linspace(-0.6, 1.6, 4401) * fr.period_kms; P_ = fr.period_kms; uc = P_ - 60.0
from scipy.special import wofz
prof = lambda x: 3.0 * np.real(wofz((x + 1j * 0.5) / 20.0)) / np.real(wofz(1j * 0.5 / 20.0))
fig, ax = plt.subplots(figsize=(18, 6.5), layout="constrained")
ax.axvspan(0, P_, color=TOL["grey"], alpha=0.18, lw=0)
for k, ls in ((0, "-"), (-1, ":"), (1, ":")):
    ax.semilogy(uu, np.maximum(prof(uu - uc - k * P_), 1e-6), ls, color=TOL["grey"] if k else EC["HI"], lw=2.5 if k == 0 else 2,
                label="its images k = ±1" if k == -1 else ("a damped piece (k = 0)" if k == 0 else None))
inside = (uu >= 0) & (uu < P_)
ax.semilogy(uu[inside], prof(np.mod(uu - uc + P_ / 2, P_) - P_ / 2)[inside], color=TOL["red"], ls="--", lw=3, label="nearest image on one period [0, P)")
for x in (0, P_):
    ax.axvline(x, color="k", lw=1.5)
ax.annotate("the red wing reappears\nat the start of the box", xy=(30, prof(np.mod(30 - uc + P_ / 2, P_) - P_ / 2)), xytext=(700, 0.5), fontsize=15,
            arrowprops=dict(arrowstyle="->"))
ax.text(P_ / 2, 2.2, "one period of the box", ha="center", fontsize=15)
ax.set(xlabel=r"$\Lambda$ [km/s]", ylabel=r"$\tau$", ylim=(1e-4, 5), xlim=(-800, P_ + 800), title="Periodic box: every piece absorbs through its nearest image")
ax.legend(loc="center"); plt.show()

*Figure — schematic of the periodic integral.* The red wing of a piece near the far face (blue) runs past Λ = P. In the periodic spectrum (red, dashed) it reappears at the start of the box, through the piece's nearest image (grey dots: the images k = ±1).

## 5. τ and F of one skewer

`SimulationSkewers.tau(ion, line_bundle, velocities)` runs the whole chain: absorbers → `optical_depth_inputs` (a `SkewerBundle` of the pieces, `Fields` of `SampledField`s and their line elements) → `OpticalDepthCalculator(line_elements=ℓ, velocity_period=P)` → τ. The calculator evaluates the pure, jitted kernel `optical_depth_kernel` in chunks of pieces and velocities, so memory stays bounded for any skewer.

Below:
- τ and F of the single skewer;
- the profiles of the individual pieces in a window: each piece adds its own Voigt profile at its own line centre;
- two consistency checks.

In [ ]:
t = time.perf_counter(); tau1 = S.tau(gp.HI, LYA, v, integ=ss.IntegrationOptions())[0]; dt = time.perf_counter() - t
F1 = np.exp(-tau1)
print(f"single skewer: tau in {dt:.1f} s | <F> = {F1.mean():.4f}, max tau = {tau1.max():.2f}, pixels with tau > 5: {np.mean(tau1 > 5):.1%}")
# per-piece profiles in a window of the path
win = (A1.n[0] > 0) & (A1.s[0] > 30.0) & (A1.s[0] < 31.0)
sub = A1.with_values(**{f: getattr(A1, f)[:, win] for f in ("s", "n", "length", "T", "v_los", "cell", "point")})
vw = v[(v > 2150) & (v < 2500)]
t_pp = ab_.tau_absorbers(sub, LYA, vw, fr, per_absorber=True, check=None)[0, 0, :win.sum()]
t_sum = ab_.tau_absorbers(sub, LYA, vw, fr, check=None)[0, 0]
check("sum of the per-piece profiles = tau of the pieces", np.abs(t_pp.sum(0) - t_sum).max() / t_sum.max(), 1e-10, "5")
# normalisation: integral of tau over one period = (pi e^2 / m_e c) f lambda x sum of the physical columns (exact for any saturation)
s_l = A1.n[0] > 0
a_p = fr.scale_factors(0, A1.s[0][s_l])
N_p = A1.n[0][s_l] * H.a ** 3 * a_p ** -2 * A1.length[0][s_l] * 3.085678e24        # n_com l a^-2 [cm^-2]
sigma0 = np.pi * 2.8179403262e-13 * 2.99792458e10                                   # pi e^2 / (m_e c) = pi r_e c [cm^2 s^-1]
lam_cm, f_osc = float(LYA.lambda0_A[0]) * 1e-8, float(LYA.f_osc[0])
pred = sigma0 * f_osc * lam_cm * N_p.sum() / 1e5                                      # [km/s]
check("integral of tau over one period = (pi e^2/m_e c) f lambda N", abs(tau1.sum() * dv / pred - 1), 1e-3, "5",
      detail=f"({tau1.sum() * dv:.3f} vs {pred:.3f} km/s)")
fig, axs = plt.subplots(3, 1, figsize=(16, 15), layout="constrained", gridspec_kw={"height_ratios": [1.3, 1.6, 1.3]})
axs[0].plot(v, F1, lw=1.2, color=EC["HI"]); axs[0].axvspan(vw[0], vw[-1], color=TOL["yellow"], alpha=0.35, lw=0)
axs[0].text(vw[-1] + 30, 0.08, "window of (b, c)", fontsize=15)
axs[0].set(xlabel=r"$\Lambda$ [km/s]", ylabel="F", title=rf"One skewer, Ly$\alpha$, over one period: $\langle F\rangle$ = {F1.mean():.3f}")
cells_w = sub.cell[0][:win.sum()]
per_cell = {c: t_pp[cells_w == c].sum(0) for c in np.unique(cells_w)}            # all the pieces of one cell
cyc = [TOL["blue"], TOL["red"], TOL["green"], TOL["yellow"], TOL["cyan"], TOL["purple"]]
for i, c in enumerate(sorted(per_cell, key=lambda c: -per_cell[c].max())[:12]):
    axs[1].semilogy(vw, np.maximum(per_cell[c], 1e-8), lw=1.6, color=cyc[i % 6], ls=("-", "--")[i // 6])
axs[1].semilogy(vw, t_sum, color="k", lw=3, label=f"sum of the {win.sum()} pieces ({len(per_cell)} cells) with 30 < s < 31 cMpc")
axs[1].plot([], [], color=TOL["grey"], lw=1.6, label="the 12 strongest cells (sum of each cell's pieces)")
axs[1].set(ylim=(1e-5, 2), ylabel=r"$\tau$", title="Every piece adds its own Voigt profile at its own line centre"); axs[1].legend(loc="lower center")
axs[2].plot(vw, F1[(v > 2150) & (v < 2500)], color=EC["HI"], **MS["default"], label="F of the whole skewer")
axs[2].plot(vw, np.exp(-t_sum), color=TOL["purple"], **MS["alternative"], label="F of the window's pieces alone")
axs[2].set(xlabel=r"$\Lambda$ [km/s]", ylabel="F", title="Gas outside the window also absorbs in it"); axs[2].legend(loc="lower right")
add_panel_letters(axs)
plt.show()

*Figure — one skewer.*
- **Top:** F over one period of the box.
- **Middle:** within the yellow window, every piece adds its full Voigt profile at its own line centre. A cell is crossed by many segments, so its pieces sit at slightly different line centres across its Hubble extent; each colour is the sum of one cell's pieces. The total (black) is τ of the window. Neighbouring cells overlap in Λ, because their thermal widths (~10–15 km/s) are comparable to their separation.
- **Bottom:** gas outside the window also absorbs in it, through peculiar velocities and line wings.

**The normalisation check** above is exact for any saturation, because τ itself, unlike F, is linear in the column. Integrated over one period, it must equal (πe²/m_ec) f λ times the sum of the physical columns of the pieces; the small difference is the ν ↔ Λ Jacobian over the far wings.

## 6. The sampling points and the resolution criterion

**What the spacing still controls.** The line elements make every column exact for any spacing (companion notebook). What the spacing controls is *where* inside its segment each piece absorbs: at its centroid, instead of across its extent in velocity, W_p = (dΛ_c/ds) ℓ_p. Replacing a top hat of width W by a delta at its centre changes the peak of a profile of width b by W²/(12 b²).

**The criterion** (default tolerance ¼): every piece with a non-negligible column must satisfy

$$W_p\le \tfrac14\, b_{V,p},$$

which bounds that error at 0.5% of the line peak. Here b_V is the width of the piece's Voigt profile: its thermal b, except for very cold gas, where the natural width takes over. Since W_p ≤ (dΛ_c/ds) w_i, the criterion is a condition on the bundle's spacing:

$$\Delta s\le \frac{b_{V,\min}}{4\,(d\Lambda_c/ds)}.$$

b ∝ (T/A)^(1/2), so the condition is the same for every line of the same ion, and a heavier ion needs finer sampling.

**Where it is checked.**
- `absorbers.sampling_check`, before every τ of deposited pieces: it reports the fraction of the column in pieces that fail.
- `OpticalDepthCalculator(resolution_tolerance=0.25)`, for any bundle of point-sampled fields: it compares consecutive line centres with ¼ of the Voigt width.

Both warn without changing τ.

Below, the single skewer is sampled from 0.125 to 16 km/s, with the same cells. For each spacing: the check, and the difference in F from the finest sampling.

In [ ]:
rows, Fs = [], {}
fr_ref = None
for ds in (0.125, 0.25, 1.0, 4.0, 16.0):
    Sd = ss.SimulationSkewers.axis_aligned(rd, SETS["single"]["cofm"], SETS["single"]["axis"], ds, cells=S.cells, name=f"single_ds{ds}", cache=CACHE)
    Ad = Sd.absorbers(gp.HI); frd = Sd.frame("exact")
    rep = ab_.sampling_check(Ad, frd, warn=False, lorentz_hwhm_kms=float(LYA.lambda0_A[0] * LYA.A21_GHz[0] / (4 * np.pi)) * 1e-4)
    Fs[ds] = np.exp(-Sd.tau(gp.HI, LYA, v, check=None)[0])
    rows.append((ds, Sd.points.shape[1], rep["max_ratio"], rep["column_fraction"], not rep["ok"]))
print(f"{'spacing [km/s]':>15s} {'points':>7s} {'max W/b':>8s} {'column with W > b/4':>20s} {'warning':>8s} {'max |F - F(0.125)|':>19s}")
for ds, n, mr, cf, nw in rows:
    print(f"{ds:15.3f} {n:7d} {mr:8.2f} {cf:20.2%} {'yes' if nw else 'no':>8s} {np.abs(Fs[ds] - Fs[0.125]).max():19.2e}")
check("default spacing (0.25 km/s) passes the criterion", rows[1][3], 1e-3, "6")
check("default spacing (0.25 km/s): max |dF| from 0.125 km/s", np.abs(Fs[0.25] - Fs[0.125]).max(), 2e-3, "6")
check("16 km/s spacing is flagged by the criterion", rows[-1][3], 1e-3, "6", kind="expect")
with warnings.catch_warnings(record=True) as wl:                    # what tau() prints when the check fails (here at 16 km/s)
    warnings.simplefilter("always")
    ab_.sampling_check(Sd.absorbers(gp.HI), Sd.frame("exact"))
print("\nthe warning at 16 km/s:", str(wl[0].message) if wl else None)
j = np.argmax(np.abs(Fs[16.0] - Fs[0.125])); wv = np.abs(v - v[j]) < 150
fig, axs = plt.subplots(2, 1, figsize=(16, 10.5), sharex=True, layout="constrained", gridspec_kw={"height_ratios": [2, 1.3]})
for ds, c, sty in ((0.125, TOL["grey"], MS["reference"]), (1.0, TOL["blue"], MS["default"]), (4.0, TOL["green"], MS["matched"]), (16.0, TOL["red"], MS["alternative"])):
    axs[0].plot(v[wv], Fs[ds][wv], color=c, **sty, label=f"spacing {ds} km/s" + (" (reference)" if ds == 0.125 else ""))
    if ds != 0.125:
        axs[1].plot(v[wv], (Fs[ds] - Fs[0.125])[wv], color=c, **sty)
residual_band(axs[1], 2e-3, label="±2×10⁻³")
axs[0].set(ylabel="F", title="Coarser sampling moves the absorption of narrow lines; 0.25 km/s is converged"); axs[0].legend(loc="lower left")
axs[1].set(xlabel=r"$\Lambda$ [km/s]", ylabel="F − F(0.125 km/s)"); axs[1].legend(loc="lower right")
add_panel_letters(axs)
plt.show()

*Figure and table — the sampling.* The column is the same at every spacing (§1), so what changes is the shape of the narrowest lines. The coldest pieces of this skewer have b ≈ 9.5 km/s. A spacing of 4 km/s already moves their absorption by 0.4 b, and the check flags a quarter of the column; at 16 km/s the error in F reaches a few 10⁻². At the default 0.25 km/s every piece passes, and F agrees with twice finer sampling to 2×10⁻⁵.

## 7. Many skewers: ⟨F⟩ and the 1D power spectrum

The same call on the 16 skewers. The P1D of δ_F = F/⟨F⟩ − 1 uses `DeltaFieldBuilder` → `OverfluxStatistics1D`, with P(k) = (Δv/N)|δ̃_k|² and k in s/km (`simulation_skewers.p1d`). The spectra are periodic, so no window is needed.

In [ ]:
Sn = SK["nset"]; frn = Sn.frame("exact"); vn, dvn = Sn.spectral_grid(RES, frn)
t = time.perf_counter(); taun = Sn.tau(gp.HI, LYA, vn); print(f"16 skewers: tau in {time.perf_counter() - t:.0f} s")
Fn = np.exp(-taun)
print(f"<F> = {Fn.mean():.4f} (per skewer {Fn.mean(1).min():.3f}-{Fn.mean(1).max():.3f}) | effective optical depth -ln<F> = {-np.log(Fn.mean()):.3f}")
k, P = ss.p1d(taun, dvn)
check("P1D: no power at negative values, finite", float(np.any(~np.isfinite(P)) or np.any(P < 0)), 0, "7")
KB = np.geomspace(k[0], k[-1] * 1.0001, 41); jb = np.digitize(k, KB) - 1; nb = np.bincount(jb, minlength=40)[:40]
kb = np.sqrt(KB[1:] * KB[:-1])[nb > 0]; Pb = (np.bincount(jb, weights=k * P / np.pi, minlength=40)[:40] / np.maximum(nb, 1))[nb > 0]
fig, axs = plt.subplots(1, 3, figsize=(22, 7), layout="constrained", gridspec_kw={"width_ratios": [2, 1, 1.2]})
for i, off, sty in ((0, 0.0, MS["default"]), (5, 1.2, MS["default"]), (8, 2.4, MS["default"])):
    axs[0].plot(vn, Fn[i] + off, color=EC["HI"], lw=1.0)
    axs[0].text(vn[-1] + 60, off + 0.5, f"skewer {i}" + (" (DLA)" if i == 8 else ""), fontsize=15, va="center")
axs[0].set(xlabel=r"$\Lambda$ [km/s]", ylabel="F (offset by 1.2)", xlim=(0, vn[-1] + 900), title="Three of the 16 skewers")
axs[1].hist(Fn.ravel(), bins=60, color=EC["HI"], histtype="step", density=True, lw=2.5)
axs[1].set(xlabel="F", ylabel="PDF", yscale="log", title="Flux PDF")
axs[2].loglog(kb, Pb, "o-", ms=7, color=EC["HI"], lw=2)
axs[2].axvspan(0.1, kb.max() * 1.1, color=TOL["grey"], alpha=0.25, lw=0); axs[2].text(0.13, 1e-11, "thermal\ncut-off", fontsize=15)
axs[2].set(xlabel="k [s/km]", ylabel=r"$kP_{\rm 1D}/\pi$", title=r"Ly$\alpha$ P1D (40 log bins)")
add_panel_letters(axs)
plt.show()

*Figure — 16 skewers.* Left: three spectra; skewer 8 crosses a damped absorber (N_HI ≈ 10^19.4 cm⁻²), whose damping wings span hundreds of km/s. Middle: the flux PDF. Right: the P1D, which falls steeply above k ≈ 0.1 s/km, where thermal broadening smooths the forest.

With only 16 skewers of 52 cMpc, these are illustrations of the pipeline, not converged statistics.

## 8. Other paths: curved and non-periodic

Nothing above assumes straight, axis-aligned or uniformly spaced skewers: the selection, the deposition, the line elements, the tangents and the frame all work on any polyline. Two tests on the same gas:
1. a "curve" of zero amplitude must reproduce the straight skewer exactly;
2. a curved skewer, x(s) = x₀ + s ẑ + A sin(2πs/λ) x̂ with A = 1.5 cMpc and λ = L/2, must converge when its sampling is halved.

The peculiar velocity is projected on the local tangent, and a(x) follows the path length. A path that is not a closed loop through the box is not periodic, so these use the direct integral (`periodic=False`) on a grid that extends 100 km/s beyond the line centres.

In [ ]:
def curved_points(A_cMpc, ds_kms, lam=L / 2):
    x0 = SETS["single"]["cofm"][0] * H.code_length_cMpc
    t_ = np.linspace(0, L, 200001)
    xyz = np.stack([x0[0] + A_cMpc * np.sin(2 * np.pi * t_ / lam), np.full_like(t_, x0[1]), t_], 1)
    arc = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(xyz, axis=0), axis=1))])
    sq = np.linspace(0, arc[-1], int(round(arc[-1] * H.velfac_kms_per_cMpc / ds_kms)) + 1)
    return np.stack([np.interp(sq, arc, xyz[:, i]) for i in range(3)], 1)[None]
DIRECT = ss.IntegrationOptions(periodic=False)
def tau_direct(pts, name):
    Sc = ss.SimulationSkewers(rd, pts, name=name, cache=CACHE)
    frc = Sc.frame("exact", periodic=False)
    vc = np.arange(-100.0, phys.c_kms * np.log(frc.a_grid[0, 0] / frc.a_grid[0, -1]) + 100.0, RES)
    return vc, Sc.tau(gp.HI, LYA, vc, integ=DIRECT)[0], Sc
v_s, t_s, _ = tau_direct(S.points, "single_direct")
v_0, t_0, S0 = tau_direct(curved_points(0.0, DS_KMS), "curve_A0")
check("zero-amplitude curve == straight skewer (points)", np.abs(S0.points - S.points).max(), 1e-12, "8")
check("zero-amplitude curve == straight skewer (tau)", np.abs(t_0 - t_s).max(), 1e-12, "8")
v_c, t_c, Sc = tau_direct(curved_points(1.5, DS_KMS), "curve_A1.5")
v_h, t_h, _ = tau_direct(curved_points(1.5, DS_KMS / 2), "curve_A1.5_half")
n_ = min(t_c.size, t_h.size)
check("curved skewer: tau converged when the sampling is halved (max |dF|)", np.abs(np.exp(-t_c[:n_]) - np.exp(-t_h[:n_])).max(), 2e-3, "8")
edge = v_s < 150
print(f"straight skewer, direct vs periodic integral within 150 km/s of the near end: max |dF| = "
      f"{np.abs(np.exp(-t_s[edge]) - np.exp(-np.interp(v_s[edge] % fr.period_kms, v, tau1, period=fr.period_kms))).max():.2f} "
      "(the gas across the face is missing without the wrap)")
fig, axs = plt.subplots(1, 2, figsize=(21, 6.5), layout="constrained", gridspec_kw={"width_ratios": [1, 2.3]})
axs[0].plot(S.points[0, :, 2], S.points[0, :, 0], color=EC["HI"], **MS["default"], label="straight")
axs[0].plot(Sc.points[0, :, 2], Sc.points[0, :, 0], color=TOL["purple"], **MS["matched"], label="curved")
axs[0].set(xlabel="z [cMpc]", ylabel="x [cMpc]", title="The two paths"); axs[0].legend(loc="upper right")
axs[1].plot(v_s, np.exp(-t_s), color=EC["HI"], **MS["default"], label="straight")
axs[1].plot(v_c, np.exp(-t_c), color=TOL["purple"], **MS["matched"], label="curved (A = 1.5 cMpc)")
axs[1].set(xlabel=r"$\Lambda$ [km/s]", ylabel="F", xlim=(0, 1500), title=r"Ly$\alpha$ along any path, with the same calls (direct integral)")
axs[1].legend(loc="lower right")
add_panel_letters(axs)
plt.show()

*Figure — a curved path through the same box.* The curved skewer crosses different gas, so its spectrum differs from the straight one. The tests establish that the same machinery reduces exactly to the straight case at zero amplitude, and converges on the curve.

The direct integral also shows what the periodic wrap of §4 provides. Near the ends of the box, the absorption of the gas just across the face is missing without it (printed above).

## 9. A metal line: Si III 1206.5 Å

Only the line and the ion density change (companion notebook): `LineBundle(["SiIII-1206"])` and `absorbers(gp.Ion("Si", 3))`. Everything in the integral is the same, with one consequence. The thermal width of Si III is 0.19 times that of H I in the same gas, so by §6 it needs ~5× finer sampling. The check below is run for both ions on skewer 8, which crosses the damped absorber with the only strong Si III system of this box.

In [ ]:
SI = gp.Ion("Si", 3); LB_SI = LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
sk8 = ss.SimulationSkewers.axis_aligned(rd, SETS["nset"]["cofm"][8:9], SETS["nset"]["axis"][8:9], DS_KMS, name="sk8", cache=CACHE)
fr8 = sk8.frame("exact"); v8, dv8 = sk8.spectral_grid(RES, fr8)
hw = lambda lb: float(lb.lambda0_A[0] * lb.A21_GHz[0] / (4 * np.pi)) * 1e-4
print(f"{'ion':>6s} {'spacing':>8s} {'b_min [km/s]':>13s} {'max W/b':>8s} {'column with W > b/4':>20s}")
for ds in (0.25, 1.0):
    Sd = sk8 if ds == DS_KMS else ss.SimulationSkewers.axis_aligned(rd, SETS["nset"]["cofm"][8:9], SETS["nset"]["axis"][8:9], ds, cells=sk8.cells)
    for ion, lb in ((gp.HI, LYA), (SI, LB_SI)):
        r = ab_.sampling_check(Sd.absorbers(ion), Sd.frame("exact"), warn=False, lorentz_hwhm_kms=hw(lb))
        print(f"{ion.species:>6s} {ds:8.2f} {r['b_min']:13.2f} {r['max_ratio']:8.2f} {r['column_fraction']:20.2%}")
        if ds == DS_KMS:
            check(f"{ion.species}: default spacing passes the criterion (skewer 8)", r["column_fraction"], 1e-3, "9")
t_hi = sk8.tau(gp.HI, LYA, v8)[0]; t_si = sk8.tau(SI, LB_SI, v8)[0]
print(f"skewer 8: max tau H I {t_hi.max():.2e}, Si III {t_si.max():.2e}; Si III equivalent width {ab_.equivalent_width_kms(t_si, dv8):.2f} km/s")
j = np.argmax(t_si); wv = np.abs(v8 - v8[j]) < 300
fig, axs = plt.subplots(2, 1, figsize=(16, 10), sharex=True, layout="constrained")
axs[0].plot(v8[wv], np.exp(-t_hi[wv]), color=EC["HI"], **MS["default"], label=r"H I Ly$\alpha$")
axs[0].set(ylabel="F", ylim=(-0.05, 1.08), title=r"Skewer 8: Ly$\alpha$ is saturated over hundreds of km/s"); axs[0].legend(loc="lower right")
axs[1].plot(v8[wv], np.exp(-t_si[wv]), color=EC["SiIII"], **MS["default"], label="Si III 1206.5")
axs[1].set(xlabel=r"$\Lambda$ [km/s]", ylabel="F", ylim=(-0.05, 1.08), title="Si III absorbs only in the dense, enriched core: sharp edges set by b ≈ 2–5 km/s")
axs[1].legend(loc="lower right")
add_panel_letters(axs)
plt.show()

*Figure — the same gas in two lines.* Lyα (top) is saturated over hundreds of km/s, through the damping wings of the absorber. Si III (bottom) absorbs only in the dense, enriched core: a saturated trough of ~40 km/s whose sharp edges are set by the few-km/s widths of its components. Those edges depend on the sampling much more than the H I profile does: at 1 km/s spacing, the check flags a third of the Si III column but none of the H I column. This is why the criterion is checked per ion.

## 10. Summary of the checks

In [ ]:
import collections, resource
print(f"{'tag':12s} {'section':7s} name")
for r in RESULTS:
    print(f"{r['tag']:12s} {r['section']:7s} {r['name']}  ({r['value']:.2e} {'>' if r['kind'] == 'expect' else '<='} {r['tol']:g}) {r['detail']}")
print("\nSUMMARY:", dict(collections.Counter(r["tag"] for r in RESULTS)), f"of {len(RESULTS)} checks")
print(f"wall time {time.perf_counter() - T_START:.0f} s | peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")